<a href="https://colab.research.google.com/github/engrjohnbosco/DATA-SCIENCE-PROJECT/blob/main/2E_Predicting_Apartment_Prices%20in%20mx_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries



In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

# Import libraries here
import warnings
from glob import glob

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge  # noqa F401
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)


In [ ]:

# Use this cell to test your wrangle function on the file `mexico-city-real-estate-1.csv`
frame1 = wrangle("data/mexico-city-real-estate-1.csv")
print("frame1 shape:", frame1.shape)
frame1.head()


In [ ]:
# Use glob to create the list files.
files = glob("data/mexico-city-real-estate-*.csv")
files


In [ ]:
# wrangle with list comprehension and concate
df = [wrangle(file) for file in files]
df = pd.concat(df, ignore_index=True)
print(df.info())
df.head()


In [ ]:
# histogram showing the distribution of apartment prices
fig, ax = plt.subplots()

# Plot the histogram on the axes object
ax.hist(df["price_aprox_usd"])
# Label axes using the axes
ax.set_xlabel("Price [$]")
ax.set_ylabel("Count")
# Add title
ax.set_title("Distribution of Apartment Prices")


In [ ]:
#  scatter plot that shows apartment price
fig, ax = plt.subplots()

# Create the scatter plot on the axes object
ax.scatter(y=df["price_aprox_usd"],x=df["surface_covered_in_m2"])

# Label axes
ax.set_xlabel("Area [sq meters]")
ax.set_ylabel("Price [USD]")

#  Add title
ax.set_title("Mexico City: Price vs. Area")


In [ ]:
# Create scatter plot of "price_usd" vs "area_m2"
plt.scatter(x=df["area_m2"], y=df["price_usd"])

# Add x-axis label

plt.xlabel("Area [sq meters]")
# Add y-axis label
plt.xlabel("Price [USD]")

# Add title
plt.title("Price vs Area")

In [ ]:

# Plot Mapbox location and price


fig = px.scatter_mapbox(
    df,  # Our DataFrame
    lat= "lat",
    lon= "lon",
    width=600,  # Width of map
  height=600,  # Height of map
    color= "price_aprox_usd",
    hover_data=["price_aprox_usd"],  # Display price when hovering mouse over house
)


fig.update_layout(mapbox_style="open-street-map")


fig.show()


In [ ]:
# Split data into feature matrix `X_train` and target vector `y_train`.

target = "price_aprox_usd"


X_train = df.drop(columns=target)
y_train = df[target]


In [ ]:
# Calculate the baseline mean absolute error for your mode
y_mean = y_train.mean()
y_pred_baseline = [y_mean] * len(y_train)
baseline_mae = mean_absolute_error(y_train, y_pred_baseline)
print("Mean apt price:", y_mean)
print("Baseline MAE:", baseline_mae)


In [ ]:
# Build Model
model =  make_pipeline(
    OneHotEncoder(use_cat_names=True),
    SimpleImputer(),
    Ridge()
)
model.fit(X_train, y_train)
# Fit model


In [ ]:
# read X_test
X_test = pd.read_csv("data/mexico-city-test-features.csv")
print(X_test.info())
X_test.head()

#Use your model to generate a Series of predictions for X_test.
y_test_pred  = pd.Series(model.predict(X_test))
y_test_pred.head()

#Create a Series named
coefficients = model.named_steps["ridge"].coef_
feature_names = model.named_steps["onehotencoder"].get_feature_names()
features = pd.Series(coefficients, index=feature_names)
feat_imp = features.sort_values(key=abs)

feat_imp


#Create a horizontal bar chart that shows the 10 most influential coefficients for your model.
fig, ax = plt.subplots()


# Create the horizontal bar plot on the axes object
feat_imp.tail(10).plot(kind="barh", ax=ax)




#  Label axes
ax.set_xlabel("Importance [USD]")
ax.set_ylabel("Feature")


# Add title
ax.set_title("Feature Importances for Apartment Price")

